# Krea 2 — RAW (52 adım) — MANUEL (ComfyUI arayüzünde elle) — Colab

ComfyUI'yi tünelle açar, grafiği **sen elle sürersin**. Uygulamanın koduna dokunmaz.

**Madde 485** *(v10 yol haritası)* — Krea 2'nin denemesi, damıtılmamış temel modelle: Krea 2 RAW, resmî **Turbo LoRA'sı** (RAW'ı 8 adımda koşmak için), Krea'nın resmî tarz LoRA'ları, Civitai'den NSFW LoRA'ları, ve **Wulver**'ın Non-Turbo hâli.

Her şey baştan iner; sonradan yeni bir şey gerekirse ComfyUI yeniden başlatılır.

**Turbo ayrı defterde: `turbo.ipynb`** — Sick Ollie de orada. İkisi ayrı runtime'da koşar, her biri kendi dosyalarını indirir. RAW için Krea 52 adım, cfg 3,5 veriyor; Turbo LoRA'sıyla 8 adım, cfg 1.

**Resmî bir RAW şablonu yok.** Bu defter Turbo'nun iki int8 şablonunu olduğu gibi Workflows paneline koyar; her birinde modeli (`unet_name`) RAW'a sen çevirirsin — `instructions.md`.

**Input:** prompt ve referans resim (ComfyUI UI'da) · **Output:** resim

**Gerekenler:** **A100** runtime. Colab **Secrets**'ta **`CIVITAI_COOKIE`** (zorunlu — NSFW LoRA'ları Civitai'den girişle iniyor) ve isteğe bağlı `HF_TOKEN`.

Sıra:
1. **CONFIG** — Civitai çerezi + GPU
2. **Ortak yardımcılar** — log + fail-loud run + model doğrulama
3. **ComfyUI** — `master`'dan; custom node yok, Krea 2 ComfyUI'nin kendinde
4. **Modeller** — ~52,7 GB; önce boş disk, sonra Civitai erişimi
5. **Workflow'lar** — resmî `image_krea2_turbo_t2i_int8` ve `image_krea2_turbo_int8_image_style_reference`, olduğu gibi; stil referansı şablonunun örnek resmi
6. **Başlat + cloudflared tünel** → UI linki

> Drive kullanılmaz; modeller her oturumda kaynaktan iner (Colab geçici diski). Üretilen resimler de orada durur: beğendiğini UI'dan indir, runtime kapanınca gider.
>
> **Runtime → Change runtime type → A100.**
>
> **Run all** → en alttaki linke gir → sol kenarda **Workflows** → bir workflow aç → `unet_name`'i RAW'a çevir → **Run** (Ctrl+Enter).
>
> Hangi dosya neden iniyor: **`indirilecekler.md`** · nasıl kullanılır: **`instructions.md`**

## 1) CONFIG

Doldurulacak bir şey yok → **Run all**.

**`CIVITAI_COOKIE`** Colab **Secrets**'tan okunur — queen-editor'ün okuduğu ad, tek sır bütün defterlere yeter. NSFW LoRA'larının yazarları indirmeyi girişe bağlamış; çerez yoksa ya da kırpılmışsa hücre burada, ilk saniyede durur, 52 GB indirdikten sonra değil. Değeri: civitai.red → giriş → F12 → Application → Cookies → `__Secure-civ-token` → değere **çift tıkla** → Ctrl+A → Ctrl+C (tek tıkla kopyalanan değer kırpık gelir). Ömrü ~30 gün. Değer hiçbir yere basılmaz.

`HF_TOKEN` da Secrets'tan okunur, değeri basılmaz. Yoksa HF indirmeleri token'sız gider: dosyalar açık, ama HF token'sız isteklere 429 dönebiliyor.

GPU burada basılır; GPU yoksa hücre durur.

**Drive mount yok.** Bu deneme Drive kullanmıyor; standardın "önce Drive mount" kuralı Drive kullanan defterler için.

In [ ]:
# === CONFIG ===
COMFY_PORT = 8188
COMFY_ROOT = "/content/ComfyUI"
COMFY_LOG  = "/content/comfyui.log"

import subprocess
from google.colab import userdata

# Required: the Civitai files' authors gate their downloads behind a login. Read here, so a missing
# or clipped cookie stops the run in its first second rather than after the Hugging Face downloads.
# The name is queen-editor's: secrets belong to the Google account, so one paste serves every notebook
# (NOTEBOOK-STANDARD, section 4). The value is never printed.
COOKIE_VALUE = (userdata.get("CIVITAI_COOKIE") or "").strip()
# Two asserts, not one: a missing secret and a clipped value are different mistakes.
assert COOKIE_VALUE, "❌ CIVITAI_COOKIE okunamadı — Colab 🔑 Secrets'a ekle ve 'Notebook access'i aç"
assert len(COOKIE_VALUE) > 200, (
    f"❌ CIVITAI_COOKIE çok kısa ({len(COOKIE_VALUE)} karakter) — değer kırpılmış: civitai.red → F12 → "
    "Application → Cookies → __Secure-civ-token, değere çift tıkla → Ctrl+A → Ctrl+C")
print("✓ CIVITAI_COOKIE: Secrets'tan okundu")

# Optional: every HF file is public, but HF can answer 429 to requests without a token. Read once here
# and handed to every download, so huggingface_hub never goes to the vault itself mid-download (in
# queen-editor that late ask timed out and H3 came down unauthenticated into a 429, madde 437).
try:
    HF_TOKEN = (userdata.get("HF_TOKEN") or "").strip()
except Exception as e:
    HF_TOKEN = ""
    print(f"⚠️  HF_TOKEN okunamadı — {type(e).__name__}: {e}\n"
          "    Token'sız inecek: dosyalar açık, ama HF 429 dönebilir.")
if HF_TOKEN:
    print("✓ HF_TOKEN: Secrets'tan okundu")

# === GPU ===
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True)
except FileNotFoundError:
    smi = None
if smi is None or smi.returncode != 0 or not smi.stdout.strip():
    said = (smi.stderr or smi.stdout).strip() if smi else "nvidia-smi bulunamadı"
    raise RuntimeError(f"❌ GPU yok — Runtime → Change runtime type → A100\n{said}")
print(f"\n=== GPU ===\n{smi.stdout.strip()}")

## 2) Ortak Yardımcılar

Tek yerde tanımlanır, sonraki hücreler bunları kullanır:

- `log(msg, level)` — konsola durum satırı
- `run(cmd, label)` — **tek komut-hatası kapısı**: exit ≠ 0 veya timeout → `RuntimeError` + komutun **gerçek stderr** kuyruğu
- `check_safetensors(path)` — **ok** / **partial** / **invalid**. Beklenen boyutu **dosyanın kendi header'ından** hesaplar (tensör `data_offsets`); sunucuya sormaz
- `head_text(path)` — dosyanın ilk byte'ları, **ham**: bozuk bir indirmede orada ne olduğunu gösterir
- `comfy_int8(path)` — int8 bir dosyanın header'ında ComfyUI'nin niceleme işaretlerinden biri var mı: katman başına `<katman>.comfy_quant` ([comfy/ops.py](https://github.com/comfyanonymous/ComfyUI/blob/master/comfy/ops.py)), ya da `__metadata__` içinde `_quantization_metadata`. İkisi de olmayan bir int8 dosyasını ComfyUI'nin nasıl okuyacağı yazılı değil, ve bir hata vermeden bozuk resim de çıkarabilir

İlk dördü kanonik defterden *(`loop_maker/comfy_ui.ipynb`, H3 ve Qwen denemeleri üzerinden)* birebir geliyor. H3'ün `strip_unreferenced_tail`'i burada yok: o damga H3'ün nicelemelerini üreten araçtan geliyordu.

In [ ]:
# === Shared helpers — log + fail-loud run + model validation ===
import os, json, time, struct, subprocess

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌", "SKIP": "⏭️ "}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def head_text(path, limit=4000):
    """First bytes of a file as raw text — the response body, printed as-is, not interpreted."""
    if not os.path.exists(path):
        return "(dosya yok)"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        text = f.read(limit).decode("utf-8", errors="replace")
    return text + (f"\n… (+{human(size - limit)})" if size > limit else "")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr."""
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def check_safetensors(path):
    """State of a model file -> ("ok" | "partial" | "invalid", msg).

    The expected total size is computed from the file itself: a safetensors file is
    [8-byte LE header length][header JSON][tensor data], and the header's data_offsets say where
    the tensor data ends. No Content-Length, no HEAD request (NOTEBOOK-STANDARD, section 3).
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    if size < 8:
        return "invalid", f"too small ({human(size)})"

    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if not (0 < header_len < 200_000_000):
            return "invalid", f"bad header length ({header_len})"
        if 8 + header_len > size:
            return "partial", f"header incomplete ({human(size)})"
        try:
            header = json.loads(f.read(header_len).decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            return "invalid", f"header parse failed ({type(e).__name__}, {human(size)})"

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:                        # metadata-only header: nothing to measure against
        return "ok", f"{human(size)}, no tensor offsets"

    expected = 8 + header_len + max(ends)
    if size == expected:
        return "ok", f"{human(size)}, {len(ends)} tensors"
    if size < expected:
        return "partial", f"{size:,} / {expected:,} bytes"
    return "invalid", f"too long: {size:,} / {expected:,} bytes"

def comfy_int8(path):
    """Whether an int8 file carries either of ComfyUI's quantisation markers: a "<layer>.comfy_quant"
    entry beside a layer's weight (comfy/ops.py), or a "_quantization_metadata" entry in the header's
    "__metadata__". What ComfyUI makes of int8 weights with neither is not documented, and a picture
    can come out broken with no error. Called on a file check_safetensors has already passed, so the
    header is whole."""
    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(header_len).decode("utf-8"))
    return ("_quantization_metadata" in (header.get("__metadata__") or {})
            or any(key.endswith(".comfy_quant") for key in header))

print("✓ Ortak yardımcılar hazır (log, run, human, head_text, check_safetensors, comfy_int8)")

## 3) ComfyUI — `master`'dan

Krea 2 **ComfyUI'nin kendinde**: `CLIPLoader`'ın türlerinden biri `krea2`; şablonların öteki node'ları (`ResolutionSelector`, `TextGenerate`, `ComfySwitchNode`, `TextEncodeQwenImageEditPlus`, …) çekirdekten. Custom node da Manager da gerekmiyor. Comfy'nin [duyurusu](https://blog.comfy.org/p/krea-2-open-source-models-are-now) en az 0.26.0 istiyor.

ComfyUI her koşuda `master`'dan çekilir, sonra klonda `nodes.py`'de `"krea2"` ve `comfy_extras/nodes_resolution.py`'de `ResolutionSelector` aranır: yoksa hücre durur — eski bir ComfyUI, grafiği açınca "missing node" olarak değil burada görünür.

`hf_xet` de burada kurulur: HF dosyaları Xet deposunda, ve `hf_hub_download` onunla parçaları doğrudan depodan, paralel çeker. Adresle (`resolve/…`) inen dosya HF'in köprüsünden geçiyor, o da sunucularının çoğunda ~8,7 MB/s'e düşüyor *(queen-editor, madde 438; xet-core #821)*.

Bir komut başarısız olursa hücre `RuntimeError` ile durur ve komutun kendi çıktısını basar.

In [ ]:
# === ComfyUI from master ===
import os

if not os.path.isdir(COMFY_ROOT):
    log("ComfyUI klonlanıyor")
    run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", COMFY_ROOT],
        "clone ComfyUI", timeout=600)
run(["git", "pull", "-q"], "git pull ComfyUI", cwd=COMFY_ROOT)
log("ComfyUI gereksinimleri kuruluyor (birkaç dakika)")
run(["pip", "install", "-q", "-r", "requirements.txt"], "pip install ComfyUI", cwd=COMFY_ROOT)
log("hf_xet kuruluyor")
run(["pip", "install", "-q", "-U", "hf_xet"], "pip install hf_xet")

# Krea 2 is in ComfyUI itself: "krea2" among CLIPLoader's types, and the templates' ResolutionSelector
# in comfy_extras. Looked for here, so a clone without them stops the run now rather than as a red
# loader or a missing node in the UI.
commit = run(["git", "log", "-1", "--format=%h %cd"], "git log", cwd=COMFY_ROOT).strip()
for path, needle in [(f"{COMFY_ROOT}/nodes.py", '"krea2"'),
                     (f"{COMFY_ROOT}/comfy_extras/nodes_resolution.py", '"ResolutionSelector"')]:
    if not os.path.exists(path) or needle not in open(path).read():
        raise RuntimeError(f"❌ Bu ComfyUI'de {needle} yok ({path}, commit {commit})")
log(f"ComfyUI hazır — {commit}, Krea 2 desteği var", "OK")

## 4) Modeller — ~52,7 GB

**Hugging Face'ten, açık.** [Comfy-Org/Krea-2](https://huggingface.co/Comfy-Org/Krea-2)'dekiler Turbo şablonlarının istediği metin kodlayıcı ve VAE, RAW modeli, resmî Turbo LoRA'sı ve Krea'nın bütün tarz LoRA'ları; Wulver yazarının deposundan, [Vaelico/Wulver](https://huggingface.co/Vaelico/Wulver). Boyutlar HF'in dosya listesinden, bayt bayt.

| Klasör | Dosya | Boyut | |
|---|---|---|---|
| `diffusion_models/` | `krea2_raw_int8_convrot.safetensors` | 13,49 GB | Krea 2 RAW (DiT), A100 için int8 |
| `text_encoders/` | `qwen3vl_4b_fp8_scaled.safetensors` | 5,24 GB | metin kodlayıcı — Krea şablonlarının istediği |
| `vae/` | `qwen_image_vae.safetensors` | 0,25 GB | VAE |
| `loras/` | `krea2_turbo_lora_rank_64_bf16.safetensors` | 0,47 GB | resmî Turbo LoRA'sı — RAW'ı 8 adımda koşmak için |
| `loras/` | `krea2_darkbrush` … `krea2_vintagetarot` (9 dosya) | 9 × 0,47 GB | Krea'nın resmî tarz LoRA'ları |
| `loras/` | `krea2_style_reference.safetensors` | 0,46 GB | referans resmin tarzı — kendi şablonuyla |
| `diffusion_models/` | `Wulver_v0.5_non_turbo_bf16.safetensors` | 25,64 GB | Wulver v0.5, Turbo LoRA'sı katılmamış hâli |

**Metin kodlayıcı neden fp8.** Krea'nın int8 şablonları `qwen3vl_4b_fp8_scaled`'i istiyor. fp8 A100'de yerel hesaplanmaz: ComfyUI ağırlıkları bf16'ya çevirip hesaplar — doğru, yerel fp8'den yavaş.

**Turbo LoRA'sı.** Comfy-Org'un README'si ne işe yaradığını yazmıyor. Wulver'ın README'si — RAW'ın bir ince ayarı — onu Non-Turbo dosyasında güç 1,0, 8 adım, cfg 1 ile kullanıyor; çıplak RAW'da aynısı denenmemiş.

**Wulver Non-Turbo neden bf16.** Non-Turbo'nun fp8'i yok, yalnız bf16 ve int8 ConvRot. Yazarı int8 ConvRot'u ComfyUI dışındaki int8 araçlarına veriyor, ve ComfyUI int8'i ancak kendi niceleme işaretinden tanıdığı için öyle bir dosyayı sessizce yanlış okuyabilir. bf16'yı ComfyUI kesin doğru okur; bedeli iki kat disk.

**int8 işareti.** Krea'nın RAW int8'ini resmî şablonlar yüklüyor, yani ComfyUI'nin niceleme işaretini taşıması beklenir; bunu görmedik. Özet dosyanın header'ına bakar ve işaret yoksa uyarır — koşu durmaz.

**Civitai'den, girişle — `loras/`.** NSFW LoRA'ları, taban modeli "Krea 2":

| Dosya | Boyut | Civitai |
|---|---|---|
| `snofs_krea_v1_5.safetensors` | 1,56 GB | [SNOFS](https://civitai.com/models/1972981), Krea 2 v1.5 |
| `krea2 standing and carry` · `krea doggy` · `krea missionary` · `krea cowgirl` · `kea2 spooning` · `kreablowjob` | 6 × 0,23 GB | [krea2 sex](https://civitai.com/models/2801881), altı pozisyon |

Önce **boş disk**: inecek dosyaların toplamı + 5 GiB pay yoksa hiçbir şey inmeden durur. Sonra **Civitai erişimi**: her dosyanın ilk 1 KB'ı çerezle istenir; çerez bayatsa hücre Civitai'nin kendi cevabıyla, ağır bir şey inmeden durur.

HF dosyaları `hf_hub_download` ile `/content/hf_stage` altına, Civitai dosyaları `curl` ile (tek bağlantı) `<ad>.part`'a iner, **doğrulanır**, sonra gerçek yerine taşınır: ComfyUI yarım bir dosyayı gerçek adıyla görmez. Doğrulama header'dan: header dosyanın boyunu zaten kanıtlıyor. HF dosyaları ayrıca HF listesindeki bayt sayısına tutulur; Civitai'de bir sürümün birincil olmayan dosyası (`fileId` ile seçilen) API'nin bayt sayısına tutulur, ötekilerde API'nin sayısı yalnız disk toplamına girer. Civitai indirmesinin toplam süre sınırı yok: hız 5 dakika boyunca 1 MiB/s'in altında kalırsa durur.

Bozuk bir HF dosyası **silinmez**, yerinde kalır ve hücre başını basıp durur. **Yeniden denemeden önce onu elle sil** (hata yolunu basar): silinmezse `hf_hub_download` aynı dosyayı yeniden verir. Civitai'nin `.part`'ı ise yeniden çalıştırınca kendiliğinden ele alınır: yarımsa kaldığı yerden devam eder, tamsa yerine konur, header'ı bozuksa başı basılıp silinir ve baştan iner. Yarıda kalan bir koşuda yerine oturmuş dosyalar atlanır.

In [ ]:
import os, glob, shutil, subprocess, time
from huggingface_hub import hf_hub_download

# === What this notebook downloads ===
# The text encoder and VAE are the ones Krea's official templates name; there is no RAW template, so
# section 5 places the two Turbo int8 ones and the user picks RAW in their unet_name. Sizes are the
# bytes HF's file list gives: the disk check adds them up, and each download is held to its own.
KREA = "Comfy-Org/Krea-2"
WULVER = "Vaelico/Wulver"
STYLES = ["darkbrush", "dotmatrix", "kidsdrawing", "neondrip", "rainywindow", "retroanime",
          "softwatercolor", "sunsetblur", "vintagetarot"]
MODELS = [
    # (repo, path in the repo, ComfyUI models folder, bytes, label)
    # int8, not fp8: fp8 and fp4 are not native on an A100.
    (KREA, "diffusion_models/krea2_raw_int8_convrot.safetensors",
     "diffusion_models", 13_492_686_496, "Krea 2 RAW DiT int8"),
    # fp8 because Krea's templates name it; ComfyUI casts it to bf16 on an A100.
    (KREA, "text_encoders/qwen3vl_4b_fp8_scaled.safetensors",
     "text_encoders", 5_242_467_968, "Qwen3-VL 4B metin kodlayıcı fp8"),
    (KREA, "vae/qwen_image_vae.safetensors",
     "vae", 253_806_246, "Qwen Image VAE"),
    # RAW takes 52 steps. Wulver's README, on its own fine-tune of RAW, runs this LoRA at 1.0 for 8 steps.
    (KREA, "loras/krea2_turbo_lora_rank_64_bf16.safetensors",
     "loras", 469_423_778, "Turbo LoRA (resmî) bf16"),
    *[(KREA, f"loras/krea2_{style}.safetensors", "loras", 469_291_992, f"Tarz LoRA'sı {style}")
      for style in STYLES],
    (KREA, "loras/krea2_style_reference.safetensors",
     "loras", 457_111_760, "Stil referansı LoRA'sı"),
    # Wulver's Non-Turbo build has no fp8 file. Its int8 one the author gives to other runtimes, and
    # ComfyUI could misread it without a word; bf16 ComfyUI reads for certain, at twice the disk.
    (WULVER, "Wulver_v0.5_non_turbo_bf16.safetensors",
     "diffusion_models", 25_640_191_128, "Wulver v0.5 Non-Turbo bf16"),
]

# Civitai: (model version, file id or None for the version's only/primary file, file name, ComfyUI
# models folder, bytes, label) -- the Turbo notebook's shape, which its checkpoint needs. Every one's
# base model is "Krea 2". The header already proves the length; the API's number (sizeKB) goes into the
# disk sum, and also holds a file picked by id (civitai_fetch).
CIVITAI = [
    (3390681, None, "snofs_krea_v1_5.safetensors", "loras", 1_562_410_216, "SNOFS Krea 2 v1.5"),
    (3170121, None, "krea2 standing and carry.safetensors", "loras", 228_587_744, "krea2 sex · carry"),
    (3158793, None, "krea doggy.safetensors", "loras", 228_587_720, "krea2 sex · dog"),
    (3169965, None, "krea missionary.safetensors", "loras", 228_587_728, "krea2 sex · Mis"),
    (3158775, None, "krea cowgirl.safetensors", "loras", 228_587_720, "krea2 sex · CG"),
    (3169634, None, "kea2 spooning.safetensors", "loras", 228_587_720, "krea2 sex · spo"),
    (3158893, None, "kreablowjob.safetensors", "loras", 228_587_720, "krea2 sex · BJ"),
]

# hf_hub_download writes the repo's folders and its own .cache under local_dir; neither belongs among
# ComfyUI's models, so a file lands here and is moved into place -- a rename on one disk, not a copy.
STAGE = "/content/hf_stage"
# GiB the disk check keeps free past the models themselves.
HEADROOM_GIB = 5

def target_of(path, folder):
    return f"{COMFY_ROOT}/models/{folder}/{os.path.basename(path)}"

def settled(path, size, label, counted_by="HF listesi"):
    """The verdict's message, or a RuntimeError showing how the file starts. `size` is the count the
    file is held to, or None when the header alone judges; `counted_by` says whose count it is. A bad
    file is never passed on and never deleted: it stays where it is, for inspection -- and since
    hf_hub_download hands a file already in STAGE straight back, and a file in place is never fetched
    again, a re-run needs it deleted by hand first."""
    state, msg = check_safetensors(path)
    if state == "ok" and size is not None and os.path.getsize(path) != size:
        state, msg = "invalid", f"{os.path.getsize(path):,} bayt, {counted_by} {size:,} diyor"
    if state != "ok":
        raise RuntimeError(f"❌ {label}: {state} — {msg}\n{path}\n--- file head ---\n{head_text(path)}\n"
                           f"Yeniden denemeden önce bu dosyayı elle sil: {path}")
    return msg

def hf_fetch(repo, path, folder, size, label):
    target = target_of(path, folder)
    if os.path.exists(target):
        log(f"{label}: zaten var ({settled(target, size, label)})")
        return
    log(f"{label}: iniyor ({human(size)})")
    start = time.perf_counter()
    try:
        # token=False, not None: None sends huggingface_hub to the vault on its own.
        got = hf_hub_download(repo, path, local_dir=STAGE, token=HF_TOKEN or False)
    except Exception as e:
        said = f"{type(e).__name__}: {e}"
        response = getattr(e, "response", None)
        if response is not None:
            try:
                body = response.text
            except Exception as read_error:   # a streamed body nobody read can raise when asked for
                body = f"(gövde okunamadı — {type(read_error).__name__}: {read_error})"
            said += f"\n--- response: HTTP {response.status_code} ---\n{body}"
        raise RuntimeError(f"❌ {label}: HF {repo}/{path}\n{said}") from None
    msg = settled(got, size, label)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    os.replace(got, target)
    took = time.perf_counter() - start
    log(f"{label}: indirildi ve doğrulandı — {took:.0f} sn, {size / took / 2**20:.1f} MB/s ({msg})", "OK")

# === Civitai (NOTEBOOK-STANDARD, section 4) ===
# civitai.red, where the cookie was issued: sent to .com it is cross-site and the answer is the login
# page. The login cookie only -- a ?token= API key authenticates as that key's account and gets 401 on
# a gated file. The cookie rides in curl's arguments, so an error prints curl's stderr, never the
# command line.
def civitai_url(version_id, file_id=None):
    """The version's own address serves its primary file; ?fileId= picks another one."""
    url = f"https://civitai.red/api/download/models/{version_id}"
    return f"{url}?fileId={file_id}" if file_id else url

def cookie_header():
    return f"Cookie: __Secure-civ-token={COOKIE_VALUE}"

PROBE = "/content/_probe.bin"

def civitai_probe(version_id, label):
    """The first KB of a Civitai file, asked with the cookie before anything heavy comes down: a stale
    or clipped cookie stops the run here, with Civitai's own answer. Always the version's bare address:
    a file selector answers 401 to the probe yet works for the download (NOTEBOOK-STANDARD, section 4),
    and the version's primary file tells whether the login reaches that version. The flags are
    queen-editor's: exit 28 is curl's timeout, which --limit-rate and --max-time can bring on a good
    probe too, so the code and the bytes decide. A safetensors file's header JSON starts at its ninth
    byte; a login page or an error JSON does not."""
    done = subprocess.run(["curl", "-sS", "-L", "--max-time", "20", "--limit-rate", "200k",
                           "-r", "0-1023", "-H", cookie_header(), "-w", "%{http_code}", "-o", PROBE,
                           civitai_url(version_id)], capture_output=True, text=True)
    if done.returncode not in (0, 28):
        raise RuntimeError(f"❌ {label}: Civitai probe — curl exit {done.returncode}\n{done.stderr.strip()}")
    code = (done.stdout or "").strip()[-3:]
    raw, said = b"", ""
    if os.path.exists(PROBE):
        with open(PROBE, "rb") as f:
            raw = f.read(10)
        said = head_text(PROBE)
        os.remove(PROBE)
    if code.startswith("2") and raw[8:10] == b'{"':
        log(f"{label}: Civitai erişimi OK", "OK")
        return
    raise RuntimeError(f"❌ {label}: Civitai probe — HTTP {code}\n"
                       f"--- Civitai yanıtı ---\n{said.strip() or '(boş gövde)'}"
                       + (f"\n--- curl ---\n{done.stderr.strip()}" if done.stderr.strip() else ""))

def civitai_fetch(version_id, file_id, filename, folder, size, label):
    # A file picked by id is held to the API's byte count as well as to its header: the header proves
    # the file is whole, the count that it is the file asked for and not the version's primary one.
    held = size if file_id else None
    target = target_of(filename, folder)
    if os.path.exists(target):
        log(f"{label}: zaten var ({settled(target, held, label, 'Civitai API')})")
        return
    url = civitai_url(version_id, file_id)
    part = target + ".part"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    # A .part an earlier run left is judged before curl touches it, as queen-editor's fetch does.
    state = None
    if os.path.exists(part):
        state, msg = check_safetensors(part)
        if state == "invalid":
            # NOTEBOOK-STANDARD section 3 keeps a bad file for inspection, but these bytes do not begin
            # with a safetensors header, so nothing appended to them can make a model of them, and
            # their head is printed right here. They go, and the file comes down fresh.
            log(f"{label}: .part bozuk ({msg}) — siliniyor, baştan iniyor\n"
                f"--- dosyanın başı ---\n{head_text(part)}", "WARN")
            os.remove(part)
        elif state == "ok":
            log(f"{label}: .part zaten tam ({msg}) — indirilmiyor")
        else:
            log(f"{label}: .part'tan devam ({msg})")
    start = time.perf_counter()
    before = os.path.getsize(part) if os.path.exists(part) else 0
    if state != "ok":
        log(f"{label}: Civitai'den iniyor")
        # One connection, by curl: Civitai redirects to its store, the B2 store answers 403 when the
        # cookie comes along, and curl drops it at the change of host where aria2c carries it. -C -
        # carries on from a partial .part. No cap on the whole transfer -- a 13 GB file from a slow
        # store outruns any fixed one; a stall stops it instead (under 1 MiB/s for 5 minutes), and
        # run()'s 4 hours is the backstop.
        try:
            run(["curl", "-sS", "-L", "-C", "-", "--fail-with-body",
                 "--speed-limit", "1048576", "--speed-time", "300",
                 "-H", cookie_header(), "-o", part, url], label, timeout=4 * 3600)
        except RuntimeError as e:
            # The .part stays: what came down before the failure may be good, and the next run judges
            # it and resumes.
            raise RuntimeError(f"❌ {e}\n{url}\n--- .part'ın başı ---\n{head_text(part)}") from None
    msg = settled(part, held, label, "Civitai API")
    os.replace(part, target)
    took, got = time.perf_counter() - start, os.path.getsize(target) - before
    log(f"{label}: indirildi ve doğrulandı — {took:.0f} sn, {got / took / 2**20:.1f} MB/s ({msg})", "OK")

# === Free disk, before anything downloads ===
need = (sum(size for _, path, folder, size, _ in MODELS if not os.path.exists(target_of(path, folder)))
        + sum(size for _, _, filename, folder, size, _ in CIVITAI
              if not os.path.exists(target_of(filename, folder))))
free = shutil.disk_usage("/content").free
log(f"İnecek: {need / 1024**3:.1f} GiB | Diskte boş: {free / 1024**3:.1f} GiB")
if free < need + HEADROOM_GIB * 1024**3:
    raise RuntimeError(f"❌ Disk yetmiyor: {need / 1024**3:.1f} GiB model + {HEADROOM_GIB} GiB pay "
                       f"gerekiyor, {free / 1024**3:.1f} GiB boş.")

# === Civitai access, before anything heavy ===
for version_id, _, filename, folder, _, label in CIVITAI:
    if not os.path.exists(target_of(filename, folder)):
        civitai_probe(version_id, label)

# === Downloads ===
for repo, path, folder, size, label in MODELS:
    hf_fetch(repo, path, folder, size, label)
for version_id, file_id, filename, folder, size, label in CIVITAI:
    civitai_fetch(version_id, file_id, filename, folder, size, label)

# === Summary (reaching here means every model downloaded + validated) ===
print("")
for folder in sorted({entry[2] for entry in MODELS} | {entry[3] for entry in CIVITAI}):
    print(f"📂 {folder}/")
    for f in sorted(glob.glob(f"{COMFY_ROOT}/models/{folder}/*.safetensors")):
        print(f"   {human(os.path.getsize(f)):>8s}  {os.path.basename(f)}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

# === ComfyUI's quantisation marks on the int8 DiT: a warning, not a stop ===
# Comfy-Org's RAW int8 is expected to carry them, since its templates load it; this shows whether it does.
for f in sorted(glob.glob(f"{COMFY_ROOT}/models/diffusion_models/*int8*.safetensors")):
    if comfy_int8(f):
        log(f"{os.path.basename(f)}: ComfyUI'nin niceleme işareti var", "OK")
    else:
        log(f"{os.path.basename(f)}: ComfyUI'nin niceleme işareti YOK — ComfyUI bu dosyayı int8 olarak "
            "okumayabilir; resim bozuk çıkarsa sebep bu olabilir", "WARN")

## 5) Workflow'lar — Workflows paneline

Krea'nın üç resmî şablonu da Turbo için ([Comfy-Org/workflow_templates](https://github.com/Comfy-Org/workflow_templates/tree/main/templates)); **resmî bir RAW şablonu yok**, ve hiçbir grafik bizim tarafımızdan yazılmaz ya da değiştirilmez. Bu defter Turbo'nun iki int8 şablonunu kaynağından **olduğu gibi** indirir ve ComfyUI'nin kullanıcı workflow klasörüne (`/content/ComfyUI/user/default/workflows/`) koyar; UI'da sol kenardaki **Workflows** panelinde açılır. Her birinde büyük node'da `unet_name`'i RAW'a ya da Wulver Non-Turbo'ya sen çevirirsin — `instructions.md`.

| Workflow | Ne için | Kaynak |
|---|---|---|
| `image_krea2_turbo_t2i_int8` | prompt'tan resim; model elle RAW'a çevrilir | [Comfy-Org, resmî](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/image_krea2_turbo_t2i_int8.json) |
| `image_krea2_turbo_int8_image_style_reference` | referans resmin tarzında resim; model elle RAW'a çevrilir | [Comfy-Org, resmî](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/image_krea2_turbo_int8_image_style_reference.json) |

Konmadan önce iki kontrol: dosya JSON mu, ve şablonun `models` listesindeki her dosya 4. bölümde inmiş mi. Tek istisna şablonun Turbo DiT'i (`krea2_turbo_int8_convrot`): bu defterde indirilmiyor, onun yerine RAW seçiliyor; hücre onu "bu defterde indirilmedi" diye yazar. Turbo DiT'i listede yoksa da hücre durur: RAW talimatı tam o yükleyicinin `unet_name`'ini çeviriyor. Listedeki öteki bir dosya eksikse hücre durur ve hangisi olduğunu söyler.

**Örnek resim.** Stil referansı şablonunun `Load Image` node'u Comfy'nin bir örnek resmini adıyla istiyor, ve [docs.comfy.org](https://docs.comfy.org/tutorials/image/krea/krea-2) adresini veriyor: [`krea2_reference_image.png`](https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/input/krea2_reference_image.png). Resim `ComfyUI/input/`'ta yoksa **Run** onu yükleyemez ve durur; o yüzden iner.

Hedefte dosya zaten varsa hiçbir şey indirilmez ve dosyaya dokunulmaz: UI'da aynı adla kaydettiğin bir grafik ya da yüklediğin bir resim, hücre tekrar çalışınca ezilmez.

In [ ]:
# === The Turbo templates -> ComfyUI's user workflows, and the style reference template's input image ===
import os, json

COMFY_TEMPLATES = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main"
# (file name, raw address): Comfy-Org's official templates, each saved as it came, none written or
# changed here. There is no RAW template; the user switches each one's unet_name to RAW or Wulver
# Non-Turbo.
TEMPLATES = [
    ("image_krea2_turbo_t2i_int8.json",
     f"{COMFY_TEMPLATES}/templates/image_krea2_turbo_t2i_int8.json"),
    ("image_krea2_turbo_int8_image_style_reference.json",
     f"{COMFY_TEMPLATES}/templates/image_krea2_turbo_int8_image_style_reference.json"),
]
# The templates' DiT is Turbo's, which this notebook does not download. It alone is let off the check
# that every model a template names is on disk -- and it must be named, since the RAW instructions
# swap exactly that loader's unet_name.
SWAPPED = ("diffusion_models", "krea2_turbo_int8_convrot.safetensors")
# The style reference template's Load Image node names this, and docs.comfy.org says where it is.
# Without it in ComfyUI/input, Run stops at the Load Image.
INPUT_IMAGES = ["krea2_reference_image.png"]
# Where ComfyUI keeps the user's saved workflows; the UI's Workflows panel lists this folder.
WORKFLOWS = f"{COMFY_ROOT}/user/default/workflows"
INPUT = f"{COMFY_ROOT}/input"
PNG = b"\x89PNG\r\n\x1a\n"

def named_models(node):
    """Every {"name", "directory"} entry in the workflow -- the loaders' own "models" lists."""
    if isinstance(node, dict):
        if isinstance(node.get("name"), str) and isinstance(node.get("directory"), str):
            yield node["directory"], node["name"]
        for value in node.values():
            yield from named_models(value)
    elif isinstance(node, list):
        for value in node:
            yield from named_models(value)

def curl_part(url, saved, label):
    """The file at `url` into saved + ".part" -- the caller checks it and renames it into place."""
    part = saved + ".part"
    try:
        run(["curl", "-sS", "-L", "--fail-with-body", "--max-time", "60", "-o", part, url],
            label, timeout=120)
    except RuntimeError as e:
        raise RuntimeError(f"❌ {e}\n{url}\n--- dosyanın başı ---\n{head_text(part)}") from None
    return part

def place_workflow(name, url):
    saved = f"{WORKFLOWS}/{name}"
    # A file already there may be the user's own save under the same name: a re-run leaves it alone.
    if os.path.exists(saved):
        log(f"Workflow zaten var, olduğu gibi bırakıldı: Workflows → {name[:-5]} ({saved})")
        return
    part = curl_part(url, saved, f"workflow {name}")
    try:
        with open(part, "rb") as f:
            graph = json.loads(f.read())
    except ValueError as e:
        raise RuntimeError(f"❌ {name}: JSON değil — {e}\n{url}\n--- dosyanın başı ---\n{head_text(part)}") from None
    wanted = sorted(set(named_models(graph)))
    missing = [f"models/{folder}/{file}" for folder, file in wanted
               if (folder, file) != SWAPPED and not os.path.exists(f"{COMFY_ROOT}/models/{folder}/{file}")]
    if SWAPPED not in wanted or missing:
        raise RuntimeError(f"❌ Workflow'un istediği dosyalar inenlerle tutmuyor ({url})\n"
                           f"İstediği: {wanted or '(model listesi bulunamadı)'}\n"
                           f"Diskte olmayan: {missing}\n"
                           f"Turbo DiT'i listede mi: {SWAPPED in wanted} (RAW talimatı onun unet_name'ini çeviriyor)")
    log(f"models/{SWAPPED[0]}/{SWAPPED[1]}: bu defterde indirilmedi — {name[:-5]}'te unet_name'i RAW'a ya da "
        "Wulver Non-Turbo'ya çevir")
    # The bytes as they came, renamed into place only after both checks.
    os.replace(part, saved)
    log(f"Workflow hazır: Workflows → {name[:-5]} ({len(wanted)} model dosyası; Turbo DiT'i dışında hepsi diskte)", "OK")

def place_image(name):
    saved = f"{INPUT}/{name}"
    # A picture already there may be one the user uploaded under the same name.
    if os.path.exists(saved):
        log(f"Örnek resim zaten var: {saved}")
        return
    url = f"{COMFY_TEMPLATES}/input/{name}"
    part = curl_part(url, saved, f"resim {name}")
    with open(part, "rb") as f:
        head = f.read(len(PNG))
    if head != PNG:
        raise RuntimeError(f"❌ {name}: PNG değil\n{url}\n--- dosyanın başı ---\n{head_text(part)}")
    os.replace(part, saved)
    log(f"Örnek resim hazır: {saved} ({human(os.path.getsize(saved))})", "OK")

os.makedirs(WORKFLOWS, exist_ok=True)
os.makedirs(INPUT, exist_ok=True)
for name, url in TEMPLATES:
    place_workflow(name, url)
for name in INPUT_IMAGES:
    place_image(name)

## 6) Başlat + cloudflared Tünel

ComfyUI arka planda başlar (90 sn içinde `/system_stats` cevap vermezse log'un son 30 satırını basıp durur), tünel linki basılır, sonra hücre **bilerek açık kalır** — biterse Colab runtime'ı boşta sayıp tüneli öldürür. ComfyUI'nin canlı log'u bu hücrede akar.

In [ ]:
import subprocess, time, urllib.request, re, os

if not os.path.isfile("/content/cloudflared"):
    run(["wget", "-q", "-O", "/content/cloudflared",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], "cloudflared")
    run(["chmod", "+x", "/content/cloudflared"], "chmod cloudflared")

# Re-run safety: kill the previous ComfyUI + tunnel before starting new ones
subprocess.run(["pkill", "-f", "main.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

logf = open(COMFY_LOG, "w")
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
                 cwd=COMFY_ROOT, stdout=logf, stderr=subprocess.STDOUT)
ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(COMFY_LOG).readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")
log(f"ComfyUI ayakta ({(i + 1) * 2}s)", "OK")

# cloudflared output goes to a file, not a pipe: an unread pipe fills up and blocks the process.
# http2, not cloudflared's default QUIC: QUIC rides on UDP, which Colab's network throttles. Queen
# Editor measured it on 2026-08-24: the same photo took 17.74 s over QUIC and 0.18 s over http2.
tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read()) if os.path.exists(tunlog) else None
    if m:
        link = m.group(0)
        break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 ComfyUI linki: {link}\n")
print("⬆️  Linke gir → sol kenarda Workflows:")
print("    image_krea2_turbo_t2i_int8                    — prompt'tan resim")
print("    image_krea2_turbo_int8_image_style_reference  — referans resmin tarzında (resim image1'e)")
print("    Her birinde unet_name → krea2_raw_int8_convrot ya da Wulver_v0.5_non_turbo_bf16 (şablonların Turbo modeli bu defterde yok)")
print("    Alt grafikte KSampler: 52 adım, cfg 3,5 — ya da Turbo LoRA'sıyla (güç 1,0) 8 adım, cfg 1")
print("    LoRA'lar, tetikler, ayarlar: instructions.md")
print("\n⚠️  cfg 3,5'te (RAW, ya da LoRA'sız Wulver Non-Turbo) şablonun negatifi ConditioningZeroOut: alt grafikte")
print("    boş bir CLIPTextEncode ekle, clip'ini CLIPLoader'dan al, KSampler'ın negative'ine ConditioningZeroOut")
print("    yerine onu bağla")
print("⚠️  t2i'de prompt_enhance açık geliyor: prompt'u düzyazıya yeniden yazar — kendi prompt'unu denemek için kapat\n")

# === Keep the cell OPEN (critical) ===
# ComfyUI + the tunnel run in the background. If this cell ENDS, Colab can call the runtime idle
# and disconnect -> ComfyUI + link die. Streaming the log keeps the cell in the foreground and
# shows generation progress when Run is pressed in the UI.
# To stop: interrupt this cell (■) or Runtime -> Disconnect.
print("📡 ComfyUI çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", COMFY_LOG])
except KeyboardInterrupt:
    log("Hücre durduruldu — ComfyUI hâlâ arka planda çalışıyor (yeni link için bu hücreyi tekrar çalıştır).", "WARN")